# Emoji Mood Translator - Colab Demo

This notebook demonstrates the same functionality as the Hugging Face Space.
Type any sentence and get back emojis that match its emotional content.

**Model:** `SamLowe/roberta-base-go_emotions` (28 emotions)

In [ ]:
!pip install -q transformers torch

In [ ]:
from transformers import pipeline
from IPython.display import display, HTML

classifier = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None
)
print("Model loaded!")

In [ ]:
EMOJI_MAP = {
    "admiration": "\U0001f929", "amusement": "\U0001f604", "anger": "\U0001f620", "annoyance": "\U0001f624",
    "approval": "\U0001f44d", "caring": "\U0001f917", "confusion": "\U0001f615", "curiosity": "\U0001f914",
    "desire": "\U0001f60d", "disappointment": "\U0001f61e", "disapproval": "\U0001f44e", "disgust": "\U0001f922",
    "embarrassment": "\U0001f633", "excitement": "\U0001f389", "fear": "\U0001f628", "gratitude": "\U0001f64f",
    "grief": "\U0001f62d", "joy": "\U0001f60a", "love": "\u2764\ufe0f", "nervousness": "\U0001f630",
    "neutral": "\U0001f610", "optimism": "\U0001f31f", "pride": "\U0001f3c6", "realization": "\U0001f4a1",
    "relief": "\U0001f60c", "remorse": "\U0001f614", "sadness": "\U0001f622", "surprise": "\U0001f632",
}

def analyze_text(text):
    results = classifier(text)[0]
    active = [r for r in results if r["score"] > 0.05]
    if len(active) < 3:
        active = sorted(results, key=lambda x: x["score"], reverse=True)[:3]
    active = sorted(active, key=lambda x: x["score"], reverse=True)[:8]
    return active

## Try It Out

Change the text in the cell below and re-run to see different emoji moods.

In [ ]:
text = "I'm so proud of what we accomplished together!"

active = analyze_text(text)

# Big emoji row
emoji_row = ''
for r in active:
    emoji = EMOJI_MAP.get(r["label"], "?")
    size = int(36 + r["score"] * 50)
    emoji_row += f'<span style="font-size:{size}px;" title="{r["label"]}">{emoji}</span> '

# Detail bars
bars = ''
for r in active:
    emoji = EMOJI_MAP.get(r["label"], "?")
    pct = r["score"] * 100
    bars += f'''
    <div style="display:flex;align-items:center;gap:8px;margin:4px 0;">
        <span style="width:24px;">{emoji}</span>
        <span style="width:110px;font-size:0.85em;color:#555;">{r["label"]}</span>
        <div style="flex:1;background:#f0f0f0;border-radius:4px;height:18px;overflow:hidden;">
            <div style="width:{pct}%;background:linear-gradient(90deg,#667eea,#764ba2);
                        border-radius:4px;height:100%;min-width:3px;"></div>
        </div>
        <span style="font-size:0.8em;color:#999;width:40px;text-align:right;">{pct:.0f}%</span>
    </div>'''

display(HTML(f'''
<div style="font-family:system-ui;max-width:500px;">
    <div style="background:#f8f8f8;padding:10px 14px;border-radius:8px;margin-bottom:12px;
                font-style:italic;color:#555;">"{text}"</div>
    <div style="text-align:center;padding:16px 0;letter-spacing:8px;">{emoji_row}</div>
    <div style="margin-top:12px;">{bars}</div>
</div>
'''))

## More Examples

In [ ]:
examples = [
    "This is confusing and frustrating.",
    "I wonder what would happen if we tried something different.",
    "Thank you so much, this means the world to me!",
    "I'm terrified of what might happen next.",
]

for text in examples:
    active = analyze_text(text)
    emojis = ' '.join(EMOJI_MAP.get(r["label"], "?") for r in active[:5])
    top = active[0]
    print(f'"{text}"')
    print(f'  {emojis}  (top: {top["label"]} {top["score"]:.0%})')
    print()